# Шаг 4: основная панельная FE-модель — HS 284690 (соединения редкоземельных металлов)

**Исследовательский вопрос:** Привела ли реформа экспортного режима Китая 2016 года к статистически значимому снижению экспорта соединений РЗМ (HS 284690) в наиболее зависимые страны-партнёры?

**Метод:** OLS с двусторонними фиксированными эффектами (фиксированные эффекты партнёра и года), переменная воздействия `Post2016 × HighExposure` (топ-15 партнёров по 2010–2014). Основной вывод использует SE, кластеризованные по партнёру; двусторонняя кластеризация (partner × year) приводится как проверка чувствительности.

**Ключевые результаты:**
- ln(стоимость): β = −3.755\*\*\* — значимый эффект по стоимостному каналу
- ln(физический объём): β = −2.309\*\*\* — значимый эффект по объёмному каналу
- ln(удельная стоимость): β = 0.007, незначим — ценовой канал отсутствует

**Входные данные:** `../data/china_ree_exports_with_controls.xlsx`  
**Выходные данные:** `../results/main_model_robust_se_hs284690.xlsx`

## 0. Импорты и конфигурация

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

INPUT_FILE  = "../data/china_ree_exports_with_controls.xlsx"
OUTPUT_FILE = "../results/main_model_robust_se_hs284690.xlsx"

SPECIAL_PARTNERS_TO_DROP = ["Other Asia, nes"]
DEP_VARS = [
    "ln_export_value",
    "ln_export_quantity",
    "ln_unit_value",
    "share_export_value_clean",
]
TREATMENT = "post2016_high_value_top15"

## 1. Вспомогательные функции

Форматирование звёздочек значимости для итоговой таблицы.

In [2]:
def stars(p):
    """Звёздочки значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def coef_with_stars(coef, p):
    if pd.isna(coef):
        return ""
    return f"{coef:.3f}{stars(p)}"

## 2. Подготовка данных и построение группы экспозиции

Загрузка панели, логарифмирование переменных, построение топ-15 группы высокой экспозиции по базовому периоду 2010–2014.

In [3]:
df = pd.read_excel(INPUT_FILE, sheet_name="panel_with_controls")
df["partner"] = df["partner"].astype(str).str.strip()
df = df[~df["partner"].isin(SPECIAL_PARTNERS_TO_DROP)].copy()

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
df["export_value_usd"] = pd.to_numeric(df["export_value_usd"], errors="coerce").fillna(0)
df["quantity_kg"]      = pd.to_numeric(df["quantity_kg"],      errors="coerce").fillna(0)

df["total_export_value_usd_clean"] = df.groupby("year")["export_value_usd"].transform("sum")
df["share_export_value_clean"] = np.where(
    df["total_export_value_usd_clean"] > 0,
    df["export_value_usd"] / df["total_export_value_usd_clean"],
    0,
)

df["ln_export_value"]    = np.log1p(df["export_value_usd"])
df["ln_export_quantity"] = np.log1p(df["quantity_kg"])
df["unit_value_usd_per_kg_clean"] = np.where(
    df["quantity_kg"] > 0, df["export_value_usd"] / df["quantity_kg"], np.nan)
df["ln_unit_value"] = np.where(
    df["unit_value_usd_per_kg_clean"] > 0,
    np.log(df["unit_value_usd_per_kg_clean"]), np.nan)

# Группа высокой экспозиции: top15 партнёров по среднему импорту 2010–2014
baseline = (
    df[df["year"].between(2010, 2014)]
    .groupby("partner")["export_value_usd"]
    .mean()
    .sort_values(ascending=False)
)
top15 = baseline.head(15).index.tolist()

df["high_value_top15"] = df["partner"].isin(top15).astype(int)
df[TREATMENT] = (df["year"] >= 2016).astype(int) * df["high_value_top15"]

top15_table = pd.DataFrame({
    "rank": range(1, len(top15) + 1),
    "partner": top15,
    "baseline_avg_export_value_usd_2010_2014": [baseline[p] for p in top15],
})

top15_table

,rank,partner,baseline_avg_export_value_usd_2010_2014
0,1,Japan,305983704.0
1,2,United States,128356808.0
2,3,France,72520878.0
3,4,"Hong Kong, China",71930618.0
4,5,Germany,34735946.0
5,6,"Korea, Rep.",33629762.0
6,7,Netherlands,25264402.0
7,8,Italy,20867540.0
8,9,Vietnam,15300894.0
9,10,Thailand,4089970.0


## 3. Оценка с шестью ковариационными матрицами

`fit_all_covariances` прогоняет одну и ту же OLS-спецификацию с шестью типами SE: nonrobust, HC1, HC3, cluster_partner, cluster_year, two_way_cluster. Это позволяет убедиться, что значимость результата не зависит от выбора SE.

In [4]:
def fit_all_covariances(df, depvar):
    """OLS с шестью типами ковариационной матрицы: nonrobust, HC1, HC3,
    cluster_partner, cluster_year, two_way_cluster. Позволяет сравнить
    чувствительность инференса к выбору SE.
    """
    data = df.dropna(subset=[depvar, TREATMENT, "partner", "year"]).copy()
    formula = f"{depvar} ~ {TREATMENT} + C(partner) + C(year)"
    model = smf.ols(formula=formula, data=data)

    fits = {
        "nonrobust":       model.fit(),
        "HC1":             model.fit(cov_type="HC1"),
        "HC3":             model.fit(cov_type="HC3"),
        "cluster_partner": model.fit(cov_type="cluster", cov_kwds={"groups": data["partner"]}),
        "cluster_year":    model.fit(cov_type="cluster", cov_kwds={"groups": data["year"]}),
    }

    # Двусторонняя кластеризация по партнёру и году как проверка чувствительности
    groups_2way = np.column_stack([
        data["partner"].astype("category").cat.codes,
        data["year"].astype("category").cat.codes,
    ])
    fits["two_way_cluster_partner_year"] = model.fit(
        cov_type="cluster", cov_kwds={"groups": groups_2way})

    rows = []
    for cov_type, res in fits.items():
        coef = res.params.get(TREATMENT, float("nan"))
        pval = res.pvalues.get(TREATMENT, float("nan"))
        rows.append({
            "depvar": depvar,
            "covariance_type": cov_type,
            "coef": coef,
            "std_error": res.bse.get(TREATMENT, float("nan")),
            "t_or_z": res.tvalues.get(TREATMENT, float("nan")),
            "p_value": pval,
            "coef_with_stars": coef_with_stars(coef, pval),
            "significance": stars(pval),
            "nobs": int(res.nobs),
            "r2_adj": res.rsquared_adj,
            "bic": res.bic,
            "rmse": float(np.sqrt(np.mean(res.resid ** 2))),
        })

    return pd.DataFrame(rows)

## 4. Итоговая таблица и сохранение

Основная таблица включает cluster_partner и two_way_cluster для каждой зависимой переменной.

In [5]:
robust_se = pd.concat(
    [fit_all_covariances(df, depvar) for depvar in DEP_VARS],
    ignore_index=True,
)

# Итоговая таблица: два основных варианта SE
final_table = robust_se[
    robust_se["covariance_type"].isin(["cluster_partner", "two_way_cluster_partner_year"])
].copy()

примечаниеs = pd.DataFrame([
    ["Цель", "Добавляет робастный инференс к основной панельной модели с фиксированными эффектами для HS 284690."],
    ["Основная модель", "Y_it ~ Post2016 x HighExposureTop15 + FE страны + FE года."],
    ["HighExposure", "Топ-15 импортных партнеров по средней стоимости экспорта в 2010-2014 годах; Other Asia, nes исключается до расчета."],
    ["Предпочтительный инференс", "В основной таблице используются кластерно-робастные SE по партнеру."],
    ["Дополнительная чувствительность", "Двусторонне кластеризованные SE по партнеру и году приводятся как дополнительная проверка робастности."],
    ["Почему не только HC-робастные SE?", "HC1/HC3 корректируют гетероскедастичность, но панельные торговые данные обычно требуют учета корреляции внутри стран во времени, поэтому кластеризация по стране более уместна."],
    ["Оговорка по интерпретации", "Результаты следует интерпретировать как эмпирическую структурную связь, а не как строгую каузальную DID-оценку."],
    ["Оговорка по ln(1+x)", "Коэффициенты нельзя механически переводить в точные процентные изменения, потому что зависимая переменная учитывает нули через ln(1+x)."],
], columns=["тема", "примечание"])

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    примечаниеs.to_excel(writer, sheet_name="methodology_примечаниеs", index=False)
    top15_table.to_excel(writer, sheet_name="top15_clean", index=False)
    robust_se.to_excel(writer, sheet_name="robust_se_comparison", index=False)
    final_table.to_excel(writer, sheet_name="final_reporting_table", index=False)

print(f"Сохранено: {OUTPUT_FILE}")
final_table

Saved: ../results/main_model_robust_se_hs284690.xlsx


,depvar,covariance_type,coef,std_error,t_or_z,p_value,coef_with_stars,significance,nobs,r2_adj,bic,rmse
3,ln_export_value,cluster_partner,-3.755139,0.506124,-7.419409,1.176444e-13,-3.755***,***,1185,0.750238,6763.764474,3.171674
5,ln_export_value,two_way_cluster_partner_year,-3.755139,0.793227,-4.734001,2.201367e-06,-3.755***,***,1185,0.750238,6763.764474,3.171674
9,ln_export_quantity,cluster_partner,-2.309396,0.447144,-5.164771,2.407332e-07,-2.309***,***,1185,0.797941,5951.633935,2.251482
11,ln_export_quantity,two_way_cluster_partner_year,-2.309396,0.492533,-4.688810,2.747986e-06,-2.309***,***,1185,0.797941,5951.633935,2.251482
15,ln_unit_value,cluster_partner,0.007496,0.212980,0.035194,9.719251e-01,0.007,,705,0.546818,2437.476214,0.880342
17,ln_unit_value,two_way_cluster_partner_year,0.007496,0.261971,0.028612,9.771737e-01,0.007,,705,0.546818,2437.476214,0.880342
21,share_export_value_clean,cluster_partner,-0.003715,0.009059,-0.410124,6.817148e-01,-0.004,,1185,0.895918,-5750.183536,0.016149
23,share_export_value_clean,two_way_cluster_partner_year,-0.003715,0.006912,-0.537512,5.909142e-01,-0.004,,1185,0.895918,-5750.183536,0.016149


## Результаты

### Итоговая таблица коэффициентов при Post2016 x HighExposure (HS 284690)

| Зависимая переменная | β | SE (кластер по партнёру) | SE (двусторонний кластер) | p |
|---|---|---|---|---|
| ln(стоимость экспорта) | **−3.755** | 0.506 | 0.793 | < 0.001 *** |
| ln(физический объём) | **−2.309** | 0.447 | 0.493 | < 0.001 *** |
| ln(удельная стоимость) | 0.007 | 0.213 | 0.262 | 0.972 (n.s.) |
| Доля в экспорте | −0.004 | 0.009 | 0.007 | 0.682 (n.s.) |

N = 1 185 (стоимость, объём, доля); N = 705 (удельная стоимость, только положительные потоки).  
Within adj. R² = 0.750 (стоимость), 0.798 (объём).

**Выводы:**

1. **Канал воздействия — объём и структура торговли, не цена.** Коэффициенты при стоимости
   и объёме значимы и отрицательны. Коэффициент при удельной стоимости практически равен нулю
   (β = 0.007) и статистически незначим — реформа не изменила экспортные цены.

2. **Устойчивость инференса.** Стандартная ошибка при кластеризации по партнёру (0.506)
   и двусторонней кластеризации по партнёру и году (0.793) дают схожий вывод о значимости.
   Двусторонняя SE более консервативна и по-прежнему даёт p < 0.001.

3. **Ограничение.** Интерпретировать как эмпирическую структурную ассоциацию:
   pre-trend тест (p = 0.0038) указывает на нарушение предположения о параллельных трендах,
   поэтому строгая каузальная интерпретация в рамках DID невозможна.
